# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #1: "Content peaks at 61-90 days, declines after 270 days" (Finding #2)
What they claim: Content follows a lifecycle — it grows for 90 days, peaks at 61-90 days, then decays after 270 days.
Where the label comes from: The "Health Score" — a FlyRank composite metric built from impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).
Does the validation carry the claim? Directionally suggestive, but the health score construction introduces circularity. Since health score includes impressions and position, pages that already perform well will score high by design. A cleaner validation would use a single external metric (e.g., raw impressions alone) to track the lifecycle, or validate the composite against a held-out performance window. The 365+ "rebound" is better read as refresh-driven recovery rather than natural age reversal — the paper notes this, which strengthens the honest framing.
Constructive suggestion: Separate the lifecycle analysis into (a) raw impression trajectory and (b) health score trajectory, then compare. If both show the same pattern, the claim would be stronger.

Finding #2: "AI-generated content is not penalized" (Myth #5)
What they claim: No blanket penalty for AI content — quality and process matter more than whether AI helped draft the page.
Where the label comes from: Age-controlled comparison of OpenAI vs Gemini model cohorts within the same publication windows.
Does the validation carry the claim? The age-controlled comparison is a good start — it removes the confound that newer content might perform differently. However, the portfolio is "almost all AI-generated," so there is no human-written baseline to establish what "penalized" would look like. The paper honestly notes that "the differences moved around rather than pointing to a single winner," which is the right way to frame an exploratory result.
Constructive suggestion: To strengthen this claim, the study would need a matched human-written cohort (same topics, same age, same length). Without that, the safer headline is: "AI content quality varies by model and editing process" rather than "AI is not penalized."

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# Recreate the target (same definition as Week 5)
visibility = df['impressions_90d'] >= df['impressions_90d'].median()
low_ctr = df['ctr'] <= df['ctr'].quantile(0.25)
low_engagement = df['engagement_rate'] <= df['engagement_rate'].quantile(0.25)
findable = df['avg_position'] <= 20
df['opportunity'] = (visibility & (low_ctr | low_engagement) & findable).astype(int)

# Grouped split: all pages from same client go to train OR test, never both
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Use client_id as the group
groups = df['client_id']

# X_all is df without opportunity and IDs
X_all = df.drop(columns=['opportunity', 'content_id', 'client_id'])
y_all = df['opportunity']

for train_idx, test_idx in gss.split(X_all, y_all, groups=groups):
    X_train_grouped = X_all.iloc[train_idx]
    X_test_grouped = X_all.iloc[test_idx]
    y_train_grouped = y_all.iloc[train_idx]
    y_test_grouped = y_all.iloc[test_idx]

print("Grouped split done.")
print("Train clients:", df.iloc[train_idx]['client_id'].nunique())
print("Test clients:", df.iloc[test_idx]['client_id'].nunique())
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train opportunity rate:", y_train_grouped.mean().round(4))
print("Test opportunity rate:", y_test_grouped.mean().round(4))

Grouped split done.
Train clients: 25
Test clients: 7
Train rows: 23837
Test rows: 6163
Train opportunity rate: 0.1872
Test opportunity rate: 0.1735


In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify column types
numeric_cols_g = X_train_grouped.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_g = X_train_grouped.select_dtypes(include=['object', 'string']).columns.tolist()

# Drop leaky features (same as Week 5 super-honest)
LEAKY = ['ctr', 'engagement_rate', 'impressions_90d', 'avg_position', 
         'impression_tier', 'position_tier', 'days_with_impressions']

numeric_cols_g = [c for c in numeric_cols_g if c not in LEAKY]
categorical_cols_g = [c for c in categorical_cols_g if c not in LEAKY]

X_train_g = X_train_grouped.drop(columns=LEAKY, errors='ignore')
X_test_g = X_test_grouped.drop(columns=LEAKY, errors='ignore')

# Preprocessing
numeric_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                          ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor_g = ColumnTransformer([
    ('num', numeric_t, numeric_cols_g),
    ('cat', categorical_t, categorical_cols_g)
])

# Train
logreg_g = Pipeline([
    ('preprocess', preprocessor_g),
    ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

logreg_g.fit(X_train_g, y_train_grouped)
y_proba_g = logreg_g.predict_proba(X_test_g)[:, 1]

print("Grouped model trained.")
print("Features:", X_train_g.shape[1])
print("Probability range:", y_proba_g.min().round(4), "to", y_proba_g.max().round(4))

Grouped model trained.
Features: 35
Probability range: 0.0 to 0.9852


In [10]:
import pandas as pd
import numpy as np

def precision_at_k(y_true, scores, k):
    top_k_idx = np.argsort(scores)[-k:][::-1]
    y_true_reset = y_true.reset_index(drop=True)
    return y_true_reset.iloc[top_k_idx].mean()

# We need to re-run the random split model on the SAME test set for fair comparison
# Or just compare using the grouped test set for both

# Let's train a random-split model on the same grouped data for fair comparison
from sklearn.model_selection import train_test_split

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X_train_g, y_train_grouped,  # Use same data as grouped split
    test_size=0.2, 
    stratify=y_train_grouped,
    random_state=42
)

# Train random split model
logreg_rand = Pipeline([
    ('preprocess', preprocessor_g),
    ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

logreg_rand.fit(X_train_rand, y_train_rand)
y_proba_rand = logreg_rand.predict_proba(X_test_rand)[:, 1]

# Now compare on comparable test sets
results_grouped = []
for k in [50, 100, 200, 500, 1000]:
    random_p = precision_at_k(y_test_rand.reset_index(drop=True), y_proba_rand, k)
    grouped_p = precision_at_k(y_test_grouped.reset_index(drop=True), y_proba_g, k)
    
    results_grouped.append({
        'K': k,
        'Base_rate': round(y_all.mean(), 4),
        'Random_split': round(random_p, 4),
        'Grouped_split': round(grouped_p, 4)
    })

results_g_df = pd.DataFrame(results_grouped)

print("\n" + "=" * 60)
print("BEFORE / AFTER: Random vs Grouped Split")
print("=" * 60)
print(f"Random split: pages randomly assigned to train/test")
print(f"Grouped split: same client's pages stay together")
print(f"Base rate: {y_all.mean():.4f}\n")
print(results_g_df.to_string(index=False))


BEFORE / AFTER: Random vs Grouped Split
Random split: pages randomly assigned to train/test
Grouped split: same client's pages stay together
Base rate: 0.1844

   K  Base_rate  Random_split  Grouped_split
  50     0.1844         0.920          0.560
 100     0.1844         0.860          0.630
 200     0.1844         0.800          0.620
 500     0.1844         0.692          0.554
1000     0.1844         0.578          0.420


### What the grouped split reveals

| Split type | K=50 precision | What it tests |
|-----------|---------------|-------------|
| Random | 92% | Can the model memorize patterns in pages it has seen? |
| Grouped | 56% | Can the model generalize to **new clients** it has never seen? |

**The 36-point drop is the honesty gap.** 

In the random split, pages from the same client can appear in both train and test. The model learns "Client X's pages look like this" and then recognizes Client X's pages in test. This is not generalization — it's memorization.

In the grouped split, the model must predict opportunities for **entirely new clients** (7 clients, 6,163 pages) based on patterns learned from **25 other clients**. This is a harder, more realistic test.

**Honest interpretation:** The model is still directional — 56% precision at K=50 is 3× the base rate. But it is not the 92% "magic" score the random split suggested. For decision support, we should use the grouped split numbers or treat the random split as an optimistic upper bound.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
print("=" * 60)
print("LEAKAGE AUDIT: Final feature set")
print("=" * 60)

print("\n--- Target construction ---")
print("opportunity = visibility AND (low_ctr OR low_engagement) AND findable")
print("  visibility: impressions_90d >= median")
print("  low_ctr: ctr <= q25")
print("  low_engagement: engagement_rate <= q25")
print("  findable: avg_position <= 20")

print("\n--- Features dropped to prevent leakage ---")
for f in LEAKY:
    print(f"  ❌ {f}")

print("\n--- Remaining features that still proxy the target ---")
proxy_features = {
    'engaged_sessions_90d': 'Proxy for engagement_rate (engaged_sessions / sessions)',
    'clicks_last_30d / impressions_last_30d': 'Proxy for CTR (different time window)',
    'days_with_sessions': 'Proxy for days_with_impressions (visibility)',
    'sessions_90d / pageviews_90d': 'Downstream of impressions (visibility proxy)',
    'scroll_rate': 'Correlated with engagement_rate'
}

for feat, reason in proxy_features.items():
    print(f"  ⚠️  {feat}: {reason}")

print("\n--- Honest assessment ---")
print("Even after dropping direct target ingredients, the model can reconstruct")
print("the target from proxy signals because search metrics are structurally")
print("interconnected: impressions → clicks → sessions → engagement.")
print("\nThis is NOT feature engineering leakage — it is DATA STRUCTURE leakage.")
print("The metrics themselves are correlated by construction in search analytics.")

LEAKAGE AUDIT: Final feature set

--- Target construction ---
opportunity = visibility AND (low_ctr OR low_engagement) AND findable
  visibility: impressions_90d >= median
  low_ctr: ctr <= q25
  low_engagement: engagement_rate <= q25
  findable: avg_position <= 20

--- Features dropped to prevent leakage ---
  ❌ ctr
  ❌ engagement_rate
  ❌ impressions_90d
  ❌ avg_position
  ❌ impression_tier
  ❌ position_tier
  ❌ days_with_impressions

--- Remaining features that still proxy the target ---
  ⚠️  engaged_sessions_90d: Proxy for engagement_rate (engaged_sessions / sessions)
  ⚠️  clicks_last_30d / impressions_last_30d: Proxy for CTR (different time window)
  ⚠️  days_with_sessions: Proxy for days_with_impressions (visibility)
  ⚠️  sessions_90d / pageviews_90d: Downstream of impressions (visibility proxy)
  ⚠️  scroll_rate: Correlated with engagement_rate

--- Honest assessment ---
Even after dropping direct target ingredients, the model can reconstruct
the target from proxy signals bec

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [12]:
print("=" * 60)
print("CLAIM REWRITE: From bold to honest")
print("=" * 60)

print("\n--- ORIGINAL (Week 5, before audit) ---")
print('"The Logistic Regression model achieves 86% precision@50,"')
print('"outperforming the rule-based baseline by 52 percentage points."')

print("\n--- PROBLEM ---")
print("This implies the model has discovered a reliable pattern.")
print("In reality, the score is inflated by data structure leakage:")
print("  - Random split allows client memorization")
print("  - Target is built from features the model can proxy")
print("  - Grouped split drops precision from 92% to 56%")

print("\n--- REWRITE (safe language) ---")
print('"The model ranks pages with low engagement relative to visibility."')
print('"On a grouped split (new clients), precision@50 is 56% —"')
print('"directionally above the 18% base rate, but not a guaranteed predictor."')
print('"Results should be treated as prioritization guidance, not ground truth."')

print("\n--- Language checklist ---")
print("✅ Observed: 'precision@50 is 56%' (measured on this split)")
print("✅ Directional: 'ranks pages with low engagement' (not 'predicts')")
print("✅ Decision-support: 'prioritization guidance' (not 'automated decisions')")
print("❌ Avoided: 'achieves', 'outperforming', 'guaranteed', 'proven'")

CLAIM REWRITE: From bold to honest

--- ORIGINAL (Week 5, before audit) ---
"The Logistic Regression model achieves 86% precision@50,"
"outperforming the rule-based baseline by 52 percentage points."

--- PROBLEM ---
This implies the model has discovered a reliable pattern.
In reality, the score is inflated by data structure leakage:
  - Random split allows client memorization
  - Target is built from features the model can proxy
  - Grouped split drops precision from 92% to 56%

--- REWRITE (safe language) ---
"The model ranks pages with low engagement relative to visibility."
"On a grouped split (new clients), precision@50 is 56% —"
"directionally above the 18% base rate, but not a guaranteed predictor."
"Results should be treated as prioritization guidance, not ground truth."

--- Language checklist ---
✅ Observed: 'precision@50 is 56%' (measured on this split)
✅ Directional: 'ranks pages with low engagement' (not 'predicts')
✅ Decision-support: 'prioritization guidance' (not 'autom

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.